# DeepFake Detection — Kaggle Training

**Architecture:** DINOv2 ViT-B/14 + FFT/DCT LightCNN + Temperature Scaling

**Avantages Kaggle vs Colab :**
- GPU P100 (16 GB VRAM) ou T4x2 — **gratuit, 30h/semaine**
- Sessions de 9h (vs 90 min Colab free)
- `/kaggle/working/` : 20 GB persistants entre sessions
- Accès direct aux datasets Kaggle

**Stratégie stockage Kaggle :**
- Code → cloné depuis GitHub
- Données → `/kaggle/working/data/` (20 GB, persist entre runs)
- Checkpoints → `/kaggle/working/checkpoints/` (persist)
- Outputs → `/kaggle/working/outputs/` (persist + téléchargeable)

| Étape | Cellule |
|-------|--------|
| Vérifier GPU + espace | §1 |
| Cloner repo | §2 |
| Installer dépendances | §3 |
| Setup dossiers | §4 |
| Télécharger images réelles | §5a |
| Télécharger fakes DF40 | §5b |
| Vérifier dataset | §5c |
| Configurer entraînement | §6 |
| **Test 5 epochs** | §7 |
| **Entraînement complet** | §7b |
| Évaluation | §8 |
| Inférence | §9 |
| Afficher résultats | §10 |

## 1. Vérifier GPU et espace disque

In [ ]:
import torch, shutil, os

# GPU
if not torch.cuda.is_available():
    raise RuntimeError("GPU non disponible !\n→ Settings > Accelerator > GPU P100")

print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Espace disque
for path, label in [('/kaggle/working', 'working (persist)'), ('/', 'total')]:
    total, used, free = shutil.disk_usage(path)
    print(f"Disque {label}: {free/1e9:.1f} GB libre / {total/1e9:.0f} GB total")

# Environnement Kaggle
print(f"\nKaggle env : {os.path.exists('/kaggle')}")
print(f"Working dir: /kaggle/working")

## 2. Cloner le repo GitHub

In [ ]:
import os

REPO_URL  = 'https://github.com/anessrb/Deepfakedetection.git'
REPO_DIR  = '/kaggle/working/Deepfakedetection'

if os.path.exists(f'{REPO_DIR}/.git'):
    print("Repo déjà cloné — mise à jour...")
    !cd {REPO_DIR} && git pull --quiet
else:
    print("Clonage...")
    !git clone --quiet {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Répertoire : {os.getcwd()}")
!git log --oneline -3

## 3. Installer les dépendances

In [ ]:
# Kaggle a déjà torch/torchvision/numpy/opencv installés
# On installe uniquement ce qui manque
!pip install -q timm albumentations einops PyYAML scipy

import torch, timm, albumentations, cv2
print(f"torch         : {torch.__version__}")
print(f"timm          : {timm.__version__}")
print(f"albumentations: {albumentations.__version__}")
print(f"opencv        : {cv2.__version__}")
print("OK.")

## 4. Setup dossiers (persistent dans /kaggle/working/)

In [ ]:
import os

REPO_DIR = '/kaggle/working/Deepfakedetection'
BASE_DIR = '/kaggle/working'

# Créer les dossiers persistants
DIRS = {
    'data'        : f'{BASE_DIR}/data',
    'checkpoints' : f'{BASE_DIR}/checkpoints',
    'outputs'     : f'{BASE_DIR}/outputs',
    'logs'        : f'{BASE_DIR}/logs',
}

for name, path in DIRS.items():
    os.makedirs(path, exist_ok=True)

# Symlinks repo → /kaggle/working/
def make_symlink(link, target):
    if os.path.islink(link):
        os.remove(link)
    elif os.path.isdir(link) and not os.listdir(link):
        os.rmdir(link)
    if not os.path.exists(link):
        os.symlink(target, link)
        return True
    return False

for name, target in DIRS.items():
    link = f'{REPO_DIR}/{name}'
    status = '✓' if make_symlink(link, target) else '~'
    print(f"  {status} {name}/ → {target}")

import shutil
_, _, free = shutil.disk_usage('/kaggle/working')
print(f"\n/kaggle/working libre : {free/1e9:.1f} GB")

## 5a. Télécharger images réelles (FF++ + CelebDF)

Source : liens officiels DF40 README  
Sauvegardé dans `/kaggle/working/data/real/` — **persist entre sessions**.

In [ ]:
from pathlib import Path
!pip install -q gdown

real_dir = Path('data/real')
real_dir.mkdir(parents=True, exist_ok=True)

REAL_SOURCES = {
    'ffpp_real'   : '1dHJdS0NZ6wpewbGA5B0PdIBS9gz28pdb',
    'celebdf_real': '1FGZ3aYsF-Yru50rPLoT5ef8-2Nkt4uBw',
}

for name, file_id in REAL_SOURCES.items():
    out_dir  = real_dir / name
    zip_path = real_dir / f'{name}.zip'

    if out_dir.exists() and (any(out_dir.rglob('*.jpg')) or any(out_dir.rglob('*.png'))):
        n = sum(1 for _ in out_dir.rglob('*.jpg')) + sum(1 for _ in out_dir.rglob('*.png'))
        print(f"  ~ {name}: déjà présent ({n:,} images)")
        continue

    print(f"  ↓ {name}...")
    !gdown {file_id} -O {str(zip_path)} --fuzzy

    if zip_path.exists():
        out_dir.mkdir(exist_ok=True)
        !unzip -q {str(zip_path)} -d {str(out_dir)}/
        zip_path.unlink()
        n = sum(1 for _ in out_dir.rglob('*.jpg')) + sum(1 for _ in out_dir.rglob('*.png'))
        print(f"    ✓ {name}: {n:,} images")
    else:
        print(f"    ✗ Échec {name}")

total_real = sum(1 for _ in real_dir.rglob('*.jpg')) + sum(1 for _ in real_dir.rglob('*.png'))
print(f"\nTotal images réelles : {total_real:,}")

## 5b. Télécharger fakes DF40 — 4 méthodes

Sauvegardé dans `/kaggle/working/data/df40/fake/` — **persist entre sessions**.

In [ ]:
from pathlib import Path
import shutil

fake_dir = Path('data/df40/fake')
fake_dir.mkdir(parents=True, exist_ok=True)

METHODS = {
    'simswap'  : '1vnEXjxgSxmiNY-RkLQdsbhayTvAAoOIc',
    'fomm'     : '1UgGDvGGw5H6Wf0KTHzjKoigHB5ALcC5Q',
    'sadtalker': '1DQCVDlFInuAH3ryQgZIyKQzPiEIyFaaa',
    'inswap'   : '1hEsN-oY9Ye2OiAzGn03UD7AajztZ6b_B',
}

for name, file_id in METHODS.items():
    out_dir  = fake_dir / name
    zip_path = fake_dir / f'{name}.zip'

    if out_dir.exists() and (any(out_dir.rglob('*.jpg')) or any(out_dir.rglob('*.png'))):
        n = sum(1 for _ in out_dir.rglob('*.jpg')) + sum(1 for _ in out_dir.rglob('*.png'))
        print(f"  ~ {name:<12}: déjà présent ({n:,} images)")
        continue

    print(f"  ↓ {name}...")
    !gdown {file_id} -O {str(zip_path)} --fuzzy

    if zip_path.exists():
        out_dir.mkdir(exist_ok=True)
        !unzip -q {str(zip_path)} -d {str(out_dir)}/
        zip_path.unlink()
        n = sum(1 for _ in out_dir.rglob('*.jpg')) + sum(1 for _ in out_dir.rglob('*.png'))
        print(f"    ✓ {name}: {n:,} images")
    else:
        print(f"    ✗ Échec {name}")

_, _, free = shutil.disk_usage('/kaggle/working')
print(f"\n/kaggle/working libre : {free/1e9:.1f} GB")

## 5c. Vérifier la structure du dataset

In [ ]:
from pathlib import Path

base     = Path('data/df40')
real_src = Path('data/real')
real_dst = base / 'real'
fake_dir = base / 'fake'

# Symlink real/ → data/real/
if real_dst.is_symlink():
    real_dst.unlink()
if not real_dst.exists() and real_src.exists():
    real_dst.symlink_to(real_src.resolve())
    print("  ✓ data/df40/real/ → data/real/")
elif real_dst.exists():
    print("  ~ data/df40/real/ existe déjà")
else:
    print("  ✗ data/real/ manquant — lance §5a")

print("\n=== Structure data/df40/ ===")
n_real = sum(1 for _ in real_dst.rglob('*.jpg')) + sum(1 for _ in real_dst.rglob('*.png')) if real_dst.exists() else 0
print(f"  real/        : {n_real:,} images")

n_fake_total = 0
if fake_dir.exists():
    for m in sorted(fake_dir.iterdir()):
        if m.is_dir():
            n = sum(1 for _ in m.rglob('*.jpg')) + sum(1 for _ in m.rglob('*.png'))
            n_fake_total += n
            print(f"  fake/{m.name:<12}: {n:,} images")

print(f"  fake/ TOTAL  : {n_fake_total:,} images")
print(f"  GRAND TOTAL  : {n_real + n_fake_total:,} images")

if n_real == 0:
    print("\n⚠ Pas d'images réelles — lance §5a")
if n_fake_total == 0:
    print("\n⚠ Pas de fakes — lance §5b")
if n_real > 0 and n_fake_total > 0:
    print("\n✓ Dataset prêt !")

## 6. Configurer l'entraînement pour Kaggle

In [ ]:
import yaml, torch

with open('configs/default.yaml', 'r') as f:
    config = yaml.safe_load(f)

vram_gb    = torch.cuda.get_device_properties(0).total_memory / 1e9
# P100 = 16 GB → batch 32 ; T4x2 = 15 GB chacun → batch 32
batch_size = 32

config['training']['batch_size']              = batch_size
config['training']['num_workers']             = 4   # Kaggle supporte plus de workers
config['training']['epochs']                  = 30
config['training']['use_amp']                 = True
config['training']['checkpoint_dir']          = 'checkpoints/'
config['datasets']['data_root']               = 'data/'
config['datasets']['max_samples_per_dataset'] = None
config['evaluation']['cross_dataset_eval']    = ['df40']
config['datasets']['df40']['root']            = 'data/df40/'

with open('configs/kaggle.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print("Config : configs/kaggle.yaml")
print(f"  GPU        : {torch.cuda.get_device_name(0)} ({vram_gb:.0f} GB VRAM)")
print(f"  batch_size : {batch_size} | epochs : 30 | AMP : True")
print(f"  checkpoints: /kaggle/working/checkpoints/ (persistent)")

## 7. Test rapide — 5 epochs (~20 min)

Vérifie que tout fonctionne avant l'entraînement complet.

In [ ]:
!python scripts/train.py \
    --config configs/kaggle.yaml \
    --epochs 5 \
    --output_dir outputs/test_run/

## 7b. Entraînement complet — 30 epochs (~2-3h sur P100)

Kaggle permet des sessions de **9h** — largement suffisant.

In [ ]:
!python scripts/train.py \
    --config configs/kaggle.yaml \
    --output_dir outputs/

In [ ]:
# Reprendre après interruption
# !python scripts/train.py \
#     --config configs/kaggle.yaml \
#     --resume checkpoints/best.pth \
#     --output_dir outputs/

## 8. Évaluation

In [ ]:
import os

for ckpt in ['outputs/detector_calibrated.pth', 'checkpoints/best.pth']:
    if os.path.exists(ckpt):
        print(f"Checkpoint : {ckpt}")
        !python scripts/evaluate.py \
            --checkpoint {ckpt} \
            --output_dir outputs/evaluation/ \
            --config configs/kaggle.yaml
        break
else:
    print("Aucun checkpoint. Lance §7 d'abord.")

## 9. Inférence

In [ ]:
# Sur Kaggle, upload via l'interface (+ bouton en haut à droite)
# puis renseigne le chemin ici
import os

TEST_IMAGE = '/kaggle/input/ton_image.jpg'  # ← modifier

ckpt = 'outputs/detector_calibrated.pth'
if not os.path.exists(ckpt):
    ckpt = 'checkpoints/best.pth'

if os.path.exists(TEST_IMAGE):
    !python scripts/inference.py \
        --input "{TEST_IMAGE}" \
        --checkpoint {ckpt} \
        --output_dir outputs/inference/
else:
    print(f"Image non trouvée : {TEST_IMAGE}")
    print("Upload une image via l'interface Kaggle ou modifie TEST_IMAGE.")

## 10. Afficher les résultats

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

plots = [
    ('Training History',  'outputs/training_history.png'),
    ('Calibration Curve', 'outputs/calibration_curve.png'),
]

available = [(t, p) for t, p in plots if Path(p).exists()]
if not available:
    print("Aucun graphique. Lance l'entraînement d'abord.")
else:
    fig, axes = plt.subplots(1, len(available), figsize=(7 * len(available), 5))
    if len(available) == 1:
        axes = [axes]
    for ax, (title, path) in zip(axes, available):
        ax.imshow(mpimg.imread(path))
        ax.set_title(title, fontsize=12)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

# Les outputs sont accessibles dans l'onglet "Output" de Kaggle
print("\nFichiers dans outputs/ :")
for f in Path('outputs').rglob('*'):
    if f.is_file():
        print(f"  {f} ({f.stat().st_size/1e6:.1f} MB)")